In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
model_a = joblib.load("model_a.pkl")
preprocessor_a = joblib.load("preprocessor.pkl")
threshold_a = joblib.load("threshold.pkl")

model_b = joblib.load("model_b.pkl")
risk_scaler_b = joblib.load("risk_scaler_b.pkl")

In [ ]:
#Load sample data
loan_df = pd.read_csv("data.csv")
momo_risk = joblib.load("user_risk.pkl")

In [6]:
print(loan_df.shape)
print(momo_risk.shape)

loan_df.head()
momo_risk.head()

(20000, 22)
(6353307, 2)


,nameOrig,behavior_risk
0,C1000000639,0.058900
1,C1000001337,0.024356
2,C1000001725,0.050281
3,C1000002591,0.083852
4,C1000003372,0.178504


In [7]:
# Get Model A probabilities
X_a = loan_df.drop(columns=["loan_paid_back"])

X_a_processed = preprocessor_a.transform(X_a)

paid_back_prob = model_a.predict_proba(X_a_processed)[:, 1]
default_prob = 1 - paid_back_prob

loan_df["default_prob"] = default_prob

loan_df[["default_prob"]].head()

,default_prob
0,0.006687
1,0.241323
2,0.046167
3,0.057326
4,0.200016


In [8]:
behavior_risk = momo_risk["behavior_risk"].values

loan_risk = loan_df[["default_prob"]].copy()

min_len = min(len(loan_risk), len(behavior_risk))

loan_risk = loan_risk.iloc[:min_len].copy()
behavior_risk = behavior_risk[:min_len]

loan_risk["behavior_risk"] = behavior_risk
loan_risk.head()

,default_prob,behavior_risk
0,0.006687,0.058900
1,0.241323,0.024356
2,0.046167,0.050281
3,0.057326,0.083852
4,0.200016,0.178504


In [10]:
loan_risk["final_risk"] = (
    0.7 * loan_risk["default_prob"] +
    0.3 * loan_risk["behavior_risk"]
)

loan_risk.head()

,default_prob,behavior_risk,final_risk
0,0.006687,0.058900,0.022351
1,0.241323,0.024356,0.176233
2,0.046167,0.050281,0.047401
3,0.057326,0.083852,0.065284
4,0.200016,0.178504,0.193563


In [ ]:
#Then add risk bands:
def risk_band(score):
    if score < 0.20:
        return "Low Risk"
    elif score < 0.40:
        return "Moderate Risk"
    elif score < 0.60:
        return "High Risk"
    else:
        return "Very High Risk"

loan_risk["risk_band"] = loan_risk["final_risk"].apply(risk_band)

loan_risk.head()

,default_prob,behavior_risk,final_risk,risk_band
0,0.006687,0.058900,0.022351,Low Risk
1,0.241323,0.024356,0.176233,Low Risk
2,0.046167,0.050281,0.047401,Low Risk
3,0.057326,0.083852,0.065284,Low Risk
4,0.200016,0.178504,0.193563,Low Risk
